# Task 3A — Random Forest for MAR Suitability + Maps + Common New Sites

Converted from `Task_3A_RandomForest.py` on 2025-12-29 08:57:59.

Run cells **top-to-bottom**. Outputs (prints/tables) are shown at each step.

## 1) Library imports

In [1]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

import rasterio
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

print("Imports OK.")
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("geopandas:", gpd.__version__)
print("rasterio:", rasterio.__version__)


Imports OK.
numpy: 1.26.4
pandas: 2.3.3
geopandas: 1.1.2
rasterio: 1.4.4


## 2) User inputs (Edit this cell)

In [2]:
DATASET = r"E:\VUB\Final\PixelDataFrames\mar_binary_dataset_all_years.csv"
REF_RASTER = r"E:\VUB\Final\AET_Clipped\AET_2014_clipped.tif"

OUT_DIR = r"E:\VUB\Final\PixelDataFrames\RF_MAR_Outputs_COMMON"
OUT_RASTER_DIR = None  # leave None to auto-create OUT_DIR\rasters

FEATURES = ["AET", "LULC", "P", "RZSM", "TEMP", "SOIL"]
TARGET = "MAR_suitable"
GROUP_COL = "pixel_id"

TRAIN_FRAC = 0.70
RANDOM_STATE = 42
NEG_POS_RATIO = 5

N_ESTIMATORS = 600
MIN_SAMPLES_LEAF = 5

PROB_THRESHOLD = 0.50
LOW_TH = 0.33
HIGH_TH = 0.66

PERSIST_THRESHOLD = 0.70

OUT_METRICS    = None
OUT_FEATIMP    = None
OUT_PRED_TABLE = None
OUT_COMMON_CSV = None
OUT_COMMON_SHP = None

print("DATASET:", DATASET)
print("REF_RASTER:", REF_RASTER)
print("OUT_DIR:", OUT_DIR)
print("FEATURES:", FEATURES)


DATASET: E:\VUB\Final\PixelDataFrames\mar_binary_dataset_all_years.csv
REF_RASTER: E:\VUB\Final\AET_Clipped\AET_2014_clipped.tif
OUT_DIR: E:\VUB\Final\PixelDataFrames\RF_MAR_Outputs_COMMON
FEATURES: ['AET', 'LULC', 'P', 'RZSM', 'TEMP', 'SOIL']


## 3) Create output folders + check inputs

In [3]:
def ensure_dir(p):
    os.makedirs(p, exist_ok=True)

if OUT_RASTER_DIR is None:
    OUT_RASTER_DIR = os.path.join(OUT_DIR, "rasters")

ensure_dir(OUT_DIR)
ensure_dir(OUT_RASTER_DIR)

OUT_METRICS    = OUT_METRICS    or os.path.join(OUT_DIR, "rf_metrics.csv")
OUT_FEATIMP    = OUT_FEATIMP    or os.path.join(OUT_DIR, "rf_feature_importance.csv")
OUT_PRED_TABLE = OUT_PRED_TABLE or os.path.join(OUT_DIR, "rf_predictions_all_records.csv")
OUT_COMMON_CSV = OUT_COMMON_CSV or os.path.join(OUT_DIR, "rf_common_new_suitable_pixels.csv")
OUT_COMMON_SHP = OUT_COMMON_SHP or os.path.join(OUT_DIR, "rf_common_new_suitable_pixels.shp")

missing = False
for p, name in [(DATASET,"DATASET"), (REF_RASTER,"REF_RASTER")]:
    if not os.path.exists(p):
        print("❌ Missing:", name, "->", p)
        missing = True
    else:
        print("✅ Found:", name)

print("✅ Output folder:", OUT_DIR)
print("✅ Raster folder :", OUT_RASTER_DIR)

print("Outputs:")
print("  OUT_METRICS   :", OUT_METRICS)
print("  OUT_FEATIMP   :", OUT_FEATIMP)
print("  OUT_PRED_TABLE:", OUT_PRED_TABLE)
print("  OUT_COMMON_CSV:", OUT_COMMON_CSV)
print("  OUT_COMMON_SHP:", OUT_COMMON_SHP)

if missing:
    raise FileNotFoundError("Fix missing paths in Config and rerun.")


✅ Found: DATASET
✅ Found: REF_RASTER
✅ Output folder: E:\VUB\Final\PixelDataFrames\RF_MAR_Outputs_COMMON
✅ Raster folder : E:\VUB\Final\PixelDataFrames\RF_MAR_Outputs_COMMON\rasters
Outputs:
  OUT_METRICS   : E:\VUB\Final\PixelDataFrames\RF_MAR_Outputs_COMMON\rf_metrics.csv
  OUT_FEATIMP   : E:\VUB\Final\PixelDataFrames\RF_MAR_Outputs_COMMON\rf_feature_importance.csv
  OUT_PRED_TABLE: E:\VUB\Final\PixelDataFrames\RF_MAR_Outputs_COMMON\rf_predictions_all_records.csv
  OUT_COMMON_CSV: E:\VUB\Final\PixelDataFrames\RF_MAR_Outputs_COMMON\rf_common_new_suitable_pixels.csv
  OUT_COMMON_SHP: E:\VUB\Final\PixelDataFrames\RF_MAR_Outputs_COMMON\rf_common_new_suitable_pixels.shp


## 4) Helper functions

In [4]:
def export_points_shp(df_in, out_shp):
    if df_in.empty:
        print(f"WARNING: Empty output; not writing: {out_shp}")
        return
    gdf = gpd.GeoDataFrame(
        df_in.copy(),
        geometry=[Point(xy) for xy in zip(df_in["lon"], df_in["lat"])],
        crs="EPSG:4326"
    )
    gdf.to_file(out_shp)
    print("Saved:", out_shp)

def prob_to_lmh_cols(prob_series, low=0.33, high=0.66):
    codes = np.full(prob_series.shape, np.nan, dtype="float32")
    labels = np.full(prob_series.shape, None, dtype=object)

    p = prob_series.to_numpy()

    m0 = np.isnan(p)
    m1 = (~m0) & (p < low)
    m2 = (~m0) & (p >= low) & (p < high)
    m3 = (~m0) & (p >= high)

    codes[m1] = 1; labels[m1] = "Low"
    codes[m2] = 2; labels[m2] = "Medium"
    codes[m3] = 3; labels[m3] = "High"

    return codes, labels

print("Helpers loaded.")


Helpers loaded.


## 5) Load + clean dataset

In [5]:
df = pd.read_csv(DATASET)

required = set(["year", "pixel_id", "lon", "lat"] + FEATURES + [TARGET])
missing_cols = [c for c in required if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing columns in dataset: {missing_cols}")

df = df.dropna(subset=FEATURES + [TARGET, "year", "pixel_id", "lon", "lat"]).copy()
df["year"] = df["year"].astype(int)
df[TARGET] = df[TARGET].astype(int)

years = sorted(df["year"].unique())
n_years = len(years)

print("Years:", years, "| N years:", n_years)
print("All records:", len(df))
print("Overall class counts:\n", df[TARGET].value_counts())
display(df.head(5))


Years: [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024] | N years: 11
All records: 513414
Overall class counts:
 MAR_suitable
0    511566
1      1848
Name: count, dtype: int64


,year,pixel_id,lon,lat,AET,LULC,P,RZSM,TEMP,SOIL,MAR_suitable
0,2014,383,27.65,70.05,206.53160,30.0,233.09563,0.270511,-0.002954,4.0,0
1,2014,384,27.75,70.05,201.74350,30.0,229.78244,0.293638,0.038191,4.0,0
2,2014,385,27.85,70.05,213.14160,30.0,229.21902,0.272055,0.080452,4.0,0
3,2014,386,27.95,70.05,228.95428,30.0,230.52446,0.347064,0.099918,4.0,0
4,2014,826,26.65,69.95,247.35196,30.0,262.63400,0.258783,-0.309810,4.0,0


## 6) Negative sampling per year (NEG_POS_RATIO × positives)

In [6]:
train_parts = []
for y in years:
    dyy = df[df["year"] == y].copy()
    pos = dyy[dyy[TARGET] == 1]
    neg = dyy[dyy[TARGET] == 0]

    n_pos = len(pos)
    if n_pos == 0:
        continue

    n_neg_need = min(len(neg), NEG_POS_RATIO * n_pos)
    if n_neg_need > 0:
        neg_sample = neg.sample(n=n_neg_need, random_state=RANDOM_STATE)
        train_parts.append(pos)
        train_parts.append(neg_sample)
    else:
        train_parts.append(pos)

train_df = pd.concat(train_parts, ignore_index=True)

print("Training records:", len(train_df))
print("Training class counts:\n", train_df[TARGET].value_counts())
display(train_df.head(5))


Training records: 11088
Training class counts:
 MAR_suitable
0    9240
1    1848
Name: count, dtype: int64


,year,pixel_id,lon,lat,AET,LULC,P,RZSM,TEMP,SOIL,MAR_suitable
0,2014,32999,27.65,62.85,565.99664,111.0,716.23865,0.310972,5.217053,4.0,1
1,2014,34304,22.25,62.55,499.93250,111.0,625.83234,0.304080,5.714527,4.0,1
2,2014,35698,25.75,62.25,485.78363,111.0,650.00620,0.318120,5.517731,4.0,1
3,2014,38431,27.25,61.65,477.28930,111.0,698.70950,0.315008,5.782207,4.0,1
4,2014,39300,23.55,61.45,544.11880,80.0,665.37850,0.377936,6.207617,4.0,1


## 7) Train/test split (grouped by pixel_id)

In [7]:
X = train_df[FEATURES]
y = train_df[TARGET]
groups = train_df[GROUP_COL]

gss = GroupShuffleSplit(n_splits=1, train_size=TRAIN_FRAC, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Train rows:", len(X_train), "| Test rows:", len(X_test))
print("Train class counts:\n", y_train.value_counts())
print("Test class counts:\n", y_test.value_counts())


Train rows: 7755 | Test rows: 3333
Train class counts:
 MAR_suitable
0    6468
1    1287
Name: count, dtype: int64
Test class counts:
 MAR_suitable
0    2772
1     561
Name: count, dtype: int64


## 8) Train Random Forest + evaluate

In [8]:
rf = RandomForestClassifier(
    n_estimators=N_ESTIMATORS,
    min_samples_leaf=MIN_SAMPLES_LEAF,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
rec = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
f1 = f1_score(y_test, y_pred, pos_label=1, zero_division=0)
roc = roc_auc_score(y_test, y_prob)

print("\n=== TEST METRICS ===")
print(f"Accuracy   : {acc:.3f}")
print(f"Precision  : {prec:.3f}")
print(f"Recall     : {rec:.3f}")
print(f"F1-score   : {f1:.3f}")
print(f"ROC-AUC    : {roc:.3f}")
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(
    y_test, y_pred,
    target_names=["Unsuitable(0)", "Suitable(1)"],
    digits=3,
    zero_division=0
))

pd.DataFrame([{
    "Accuracy": acc,
    "Precision": prec,
    "Recall": rec,
    "F1_score": f1,
    "ROC_AUC": roc,
    "Train_frac": TRAIN_FRAC,
    "NEG_POS_RATIO": NEG_POS_RATIO,
    "n_estimators": N_ESTIMATORS,
    "min_samples_leaf": MIN_SAMPLES_LEAF,
    "prob_threshold": PROB_THRESHOLD,
    "persist_threshold": PERSIST_THRESHOLD,
    "random_state": RANDOM_STATE
}]).to_csv(OUT_METRICS, index=False)

fi = pd.DataFrame({"feature": FEATURES, "importance_mdi": rf.feature_importances_}).sort_values("importance_mdi", ascending=False)
fi.to_csv(OUT_FEATIMP, index=False)

print("\nSaved metrics:", OUT_METRICS)
print("Saved feature importance:", OUT_FEATIMP)
display(fi)



=== TEST METRICS ===
Accuracy   : 0.853
Precision  : 0.568
Recall     : 0.526
F1-score   : 0.546
ROC-AUC    : 0.849

Confusion Matrix:
 [[2548  224]
 [ 266  295]]

Classification Report:
                precision    recall  f1-score   support

Unsuitable(0)      0.905     0.919     0.912      2772
  Suitable(1)      0.568     0.526     0.546       561

     accuracy                          0.853      3333
    macro avg      0.737     0.723     0.729      3333
 weighted avg      0.849     0.853     0.851      3333


Saved metrics: E:\VUB\Final\PixelDataFrames\RF_MAR_Outputs_COMMON\rf_metrics.csv
Saved feature importance: E:\VUB\Final\PixelDataFrames\RF_MAR_Outputs_COMMON\rf_feature_importance.csv


,feature,importance_mdi
4,TEMP,0.294705
3,RZSM,0.214554
0,AET,0.193528
1,LULC,0.126533
2,P,0.123640
5,SOIL,0.047039


## 9) Predict for all pixels + LMH reclass

In [9]:
df["MAR_probability"] = rf.predict_proba(df[FEATURES])[:, 1]
df["MAR_predicted"] = (df["MAR_probability"] >= PROB_THRESHOLD).astype(int)

df["suit_class_code"], df["suit_class"] = prob_to_lmh_cols(df["MAR_probability"], LOW_TH, HIGH_TH)

df.to_csv(OUT_PRED_TABLE, index=False)
print("Saved prediction table:", OUT_PRED_TABLE)

display(df[["year","pixel_id","MAR_probability","MAR_predicted","suit_class_code","suit_class"]].head(10))


Saved prediction table: E:\VUB\Final\PixelDataFrames\RF_MAR_Outputs_COMMON\rf_predictions_all_records.csv


,year,pixel_id,MAR_probability,MAR_predicted,suit_class_code,suit_class
0,2014,383,0.023891,0,1.0,Low
1,2014,384,0.018111,0,1.0,Low
2,2014,385,0.023891,0,1.0,Low
3,2014,386,0.014928,0,1.0,Low
4,2014,826,0.023152,0,1.0,Low
5,2014,828,0.023891,0,1.0,Low
6,2014,833,0.013308,0,1.0,Low
7,2014,834,0.005963,0,1.0,Low
8,2014,835,0.004787,0,1.0,Low
9,2014,836,0.005483,0,1.0,Low


## 10) Common/persistent new suitable sites (CSV + SHP)
Sites appearing more than 70% (8 years) of time suitable in past 11 years

In [11]:
new_all = df[(df[TARGET] == 0) & (df["MAR_predicted"] == 1)].copy()

if new_all.empty:
    print("WARNING: No new suitable records found. Common output will be empty.")
    pd.DataFrame().to_csv(OUT_COMMON_CSV, index=False)
else:
    common_summary = (
        new_all.groupby("pixel_id")
              .agg(
                  years_new_suitable=("MAR_predicted", "sum"),
                  mean_probability=("MAR_probability", "mean"),
                  lon=("lon", "first"),
                  lat=("lat", "first"),
                  AET=("AET", "mean"),
                  P=("P", "mean"),
                  RZSM=("RZSM", "mean"),
                  TEMP=("TEMP", "mean"),
                  LULC=("LULC", "first"),
                  SOIL=("SOIL", "first"),
              )
              .reset_index()
    )
    common_summary["years_total"] = n_years
    common_summary["new_suitable_ratio"] = common_summary["years_new_suitable"] / n_years

    common = common_summary[common_summary["new_suitable_ratio"] >= PERSIST_THRESHOLD].copy()
    common = common.sort_values(["new_suitable_ratio", "mean_probability"], ascending=False)

    common["suit_class_code"], common["suit_class"] = prob_to_lmh_cols(common["mean_probability"], LOW_TH, HIGH_TH)

    common.to_csv(OUT_COMMON_CSV, index=False)
    print("Saved common new suitable CSV:", OUT_COMMON_CSV)
    print("Common new suitable pixels:", len(common))
    display(common.head(20))

    export_points_shp(common, OUT_COMMON_SHP)


Saved common new suitable CSV: E:\VUB\Final\PixelDataFrames\RF_MAR_Outputs_COMMON\rf_common_new_suitable_pixels.csv
Common new suitable pixels: 1353


,pixel_id,years_new_suitable,mean_probability,lon,lat,AET,P,RZSM,TEMP,LULC,SOIL,years_total,new_suitable_ratio,suit_class_code,suit_class
4652,87607,11,0.967776,7.15,50.75,596.094782,936.702650,0.295152,11.388776,50.0,3.0,11,1.0,3.0,High
4546,87140,11,0.961496,5.75,50.85,615.800195,918.408874,0.285482,11.380682,50.0,3.0,11,1.0,3.0,High
4448,86688,11,0.960864,5.85,50.95,602.628765,862.489728,0.288145,11.644835,50.0,3.0,11,1.0,3.0,High
3160,81696,11,0.956574,4.95,52.05,628.518620,890.502617,0.489828,11.374297,40.0,3.0,11,1.0,3.0,High
4447,86687,11,0.955468,5.75,50.95,588.278555,870.686525,0.328304,11.643363,50.0,3.0,11,1.0,3.0,High
6249,94876,11,0.953804,9.25,49.15,616.111619,888.689830,0.285308,11.424546,50.0,3.0,11,1.0,3.0,High
4549,87143,11,0.952534,6.05,50.85,621.917311,921.915876,0.289710,11.210023,50.0,3.0,11,1.0,3.0,High
3880,84434,11,0.945664,6.95,51.45,654.971525,1042.349403,0.310667,11.341930,50.0,3.0,11,1.0,3.0,High
4733,88045,11,0.945601,5.65,50.65,604.016391,1006.430811,0.280303,10.980099,50.0,3.0,11,1.0,3.0,High
4224,85791,11,0.941619,6.75,51.15,632.811875,975.229185,0.308888,11.759259,50.0,3.0,11,1.0,3.0,High


Saved: E:\VUB\Final\PixelDataFrames\RF_MAR_Outputs_COMMON\rf_common_new_suitable_pixels.shp


C:\Users\tejue\AppData\Local\Temp\ipykernel_17768\4202261359.py:10: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(out_shp)
C:\Users\tejue\anaconda3\lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'years_new_suitable' to 'years_new_'
  ogr_write(
C:\Users\tejue\anaconda3\lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'mean_probability' to 'mean_proba'
  ogr_write(
C:\Users\tejue\anaconda3\lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'years_total' to 'years_tota'
  ogr_write(
C:\Users\tejue\anaconda3\lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'new_suitable_ratio' to 'new_suitab'
  ogr_write(
C:\Users\tejue\anaconda3\lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'suit_class_code' to 'suit_class'
  ogr_write(
C:\Users\te

## 11) Export year-wise probability + LMH rasters

In [12]:
with rasterio.open(REF_RASTER) as ref:
    profile = ref.profile.copy()
    width, height = ref.width, ref.height

df["row"] = (df["pixel_id"].astype("int64") // width).astype("int32")
df["col"] = (df["pixel_id"].astype("int64") % width).astype("int32")

prob_profile = profile.copy()
prob_profile.update(dtype="float32", count=1, nodata=np.nan, compress="lzw")

cls_profile = profile.copy()
cls_profile.update(dtype="uint8", count=1, nodata=0, compress="lzw")

for y in years:
    dyy = df[df["year"] == y]

    prob_arr = np.full((height, width), np.nan, dtype="float32")
    cls_arr = np.zeros((height, width), dtype="uint8")

    rr = dyy["row"].to_numpy()
    cc = dyy["col"].to_numpy()
    pp = dyy["MAR_probability"].to_numpy(dtype="float32")

    m = (rr >= 0) & (rr < height) & (cc >= 0) & (cc < width)
    rr, cc, pp = rr[m], cc[m], pp[m]

    prob_arr[rr, cc] = pp

    valid = ~np.isnan(prob_arr)
    cls_arr[valid & (prob_arr < LOW_TH)] = 1
    cls_arr[valid & (prob_arr >= LOW_TH) & (prob_arr < HIGH_TH)] = 2
    cls_arr[valid & (prob_arr >= HIGH_TH)] = 3

    prob_tif = os.path.join(OUT_RASTER_DIR, f"suitability_probability_{y}.tif")
    cls_tif  = os.path.join(OUT_RASTER_DIR, f"suitability_class_LMH_{y}.tif")

    with rasterio.open(prob_tif, "w", **prob_profile) as dst:
        dst.write(prob_arr, 1)

    with rasterio.open(cls_tif, "w", **cls_profile) as dst:
        dst.write(cls_arr, 1)

    print(f"Saved rasters for {y}: {os.path.basename(prob_tif)} , {os.path.basename(cls_tif)}")

print("DONE. All outputs saved in:", OUT_DIR)


Saved rasters for 2014: suitability_probability_2014.tif , suitability_class_LMH_2014.tif
Saved rasters for 2015: suitability_probability_2015.tif , suitability_class_LMH_2015.tif
Saved rasters for 2016: suitability_probability_2016.tif , suitability_class_LMH_2016.tif
Saved rasters for 2017: suitability_probability_2017.tif , suitability_class_LMH_2017.tif
Saved rasters for 2018: suitability_probability_2018.tif , suitability_class_LMH_2018.tif
Saved rasters for 2019: suitability_probability_2019.tif , suitability_class_LMH_2019.tif
Saved rasters for 2020: suitability_probability_2020.tif , suitability_class_LMH_2020.tif
Saved rasters for 2021: suitability_probability_2021.tif , suitability_class_LMH_2021.tif
Saved rasters for 2022: suitability_probability_2022.tif , suitability_class_LMH_2022.tif
Saved rasters for 2023: suitability_probability_2023.tif , suitability_class_LMH_2023.tif
Saved rasters for 2024: suitability_probability_2024.tif , suitability_class_LMH_2024.tif
DONE. All 